# Lab 6 — DAGs, Colliders and Defensible Identification
# المعمل 6 — الرسوم البيانية السببية والمُربِكات وتحديد الأثر بشكل قابل للدفاع

**Module 6 — Causal Graphs and Confounding**
**الوحدة 6 — الرسوم البيانية السببية والمُربِكات**

*SDA-DSC-213 · SDAIA Academy · Day 3, Hour 5 · 50 minutes*

---

## Why this lab exists

In Lab 5a you adjusted for age, digital literacy, prior usage, device age and region — and
you *did not* adjust for `support_calls`. In Lab 5b you were told never to control for
`reminders_sent`.

**On what basis?** So far: instinct and a rule of thumb. This lab supplies the machinery that
turns both of those choices into derivations.

A **DAG** (*رسم بياني موجّه غير دوري*) is a picture of your causal assumptions. Nodes are
variables; arrows are direct causal effects; and the **absence** of an arrow is itself a
strong claim. The payoff is mechanical: once the graph is drawn, the adjustment set is
*derived*, not argued about.

The module's most counter-intuitive and most valuable lesson:

> **Controlling for the wrong variable creates bias.** More controls is not more rigour.

| | |
|---|---|
| **Objective** | Build and defend a DAG, derive the adjustment set with the backdoor criterion, reproduce the collider trap, run identify→estimate→refute, and issue an identification verdict |
| **Data** | `injaz_users.csv` (20,000 rows), `collider_sim.parquet` (50,000 rows) |
| **Ground truth** | collider correlation **0.00 → −0.20** on conditioning; training ATT **+6.0 pp** |

### Bilingual key terms (first use)

| English | العربية |
|---|---|
| Directed acyclic graph (DAG) | رسم بياني موجّه غير دوري |
| Confounder / fork | مُربِك |
| Mediator / chain | وسيط |
| Collider | مُصادِم |
| Backdoor criterion | معيار الباب الخلفي |
| d-separation | الفصل الاتجاهي |


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
# --- lab scaffolding: self-checks that never crash the notebook ----------
def check(name, fn):
    """Run a self-check. Prints PASS/FAIL/TODO instead of raising, so the
    notebook always runs top-to-bottom even with unfinished exercises."""
    try:
        fn()
    except AssertionError as e:
        print(f"[ TODO ]  {name}: {e}")
        return False
    except Exception as e:
        print(f"[ ERROR ] {name}: {type(e).__name__}: {e}")
        return False
    print(f"[  OK  ]  {name}")
    return True


def needs(*objs, msg="complete the TODO cell above first"):
    """Guard downstream cells so an unfinished TODO skips rather than crashes."""
    if any(o is None for o in objs):
        print(f"[ SKIPPED ] {msg}")
        return False
    return True

print("Scaffolding loaded: use check(...) to self-test and needs(...) to guard.")

## Step 1 — Draw the DAG, and justify every arrow *(8 min)*

The question: **does the digital-literacy training programme raise service completion?**

Every arrow is an assumption you must be able to defend in one line. Every *missing* arrow is
also an assumption — a stronger one.

| Edge | Justification |
|---|---|
| `digital_literacy → training` | more confident users are far likelier to sign up |
| `digital_literacy → completed` | more confident users complete more, programme or not |
| `device_age → training` | users on older devices struggle more and seek help |
| `device_age → completed` | older devices fail more often mid-form |
| `training → support_calls` | the programme changes how often people call support |
| `support_calls → completed` | a support call can rescue a stuck submission |
| `training → completed` | **the effect we want to measure** |

Note what is *not* here: no arrow from `support_calls` back into `training` (calls happen
after enrolment) and no arrow from `completed` into anything (it is the endpoint).

In [ ]:
# TODO: build the DAG as a networkx.DiGraph from the edge table above, then wrap it in
#       InjazDAG(graph, treatment="training", outcome="completed").
#       Store the graph in `G` and the wrapper in `dag`.
import networkx as nx
from causal_utils import InjazDAG, draw_dag, NAVY, BLUE, ORANGE, TEAL, SLATE

G = None      # <-- replace
dag = None    # <-- replace

def _c():
    assert G is not None and dag is not None, "G / dag still None"
    assert G.number_of_edges() == 7, f"expected 7 edges, got {G.number_of_edges()}"
    assert nx.is_directed_acyclic_graph(G), "your graph has a cycle - it is not a DAG"
check("DAG built with 7 edges and no cycles", _c)

## Step 2 — Derive the adjustment set by hand, then check *(8 min)*

**Backdoor paths** are non-causal paths from *T* to *Y* that begin with an arrow *into T*.
Each one is a route for confounding and must be blocked.

Trace them yourself first, on paper:

```
training <- digital_literacy -> completed
training <- device_age       -> completed
```

Both are **forks**. Conditioning on the middle node blocks them. So the candidate adjustment
set is `{digital_literacy, device_age}`.

**Why `support_calls` is excluded**, formally: the backdoor criterion requires the adjustment
set to contain **no descendant of the treatment**. `support_calls` is a direct child of
`training`. It is on a *causal* path (`training → support_calls → completed`), so blocking it
would remove part of the very effect you are measuring. It is not a confounder; it is a
**mediator** (*وسيط*).

In [ ]:
# TODO: print dag.explain(), then compute:
#         all_sets   = backdoor_adjustment_sets(G, "training", "completed")
#         minimal    = dag.minimal_adjustment_set()
#       and verify with is_valid_adjustment_set(...) that adding support_calls
#       makes the set INVALID. Store minimal in `minimal_set`.
from causal_utils import backdoor_adjustment_sets, is_valid_adjustment_set

minimal_set = None    # <-- replace

def _c():
    assert minimal_set is not None, "minimal_set is still None"
    assert minimal_set == {"digital_literacy", "device_age"}, \
        f"expected {{digital_literacy, device_age}}, got {minimal_set}"
    assert not is_valid_adjustment_set(G, "training", "completed",
                                       minimal_set | {"support_calls"}), \
        "adding the mediator should make the set invalid"
check("backdoor set derived; mediator correctly rejected", _c)

## Step 3 — The three junctions *(5 min)*

Every path in any DAG is built from three junction types. Their behaviour under conditioning
is the entire content of Module 6.

| Structure | Shape | Path is open… | Conditioning on the middle node… |
|---|---|---|---|
| **Chain (mediator)** | X → M → Y | when M is *not* conditioned | **blocks** it (removes the mediated effect) |
| **Fork (confounder)** | X ← C → Y | when C is *not* conditioned | **blocks** the spurious association — this is what you want |
| **Collider** | X → K ← Y | **only when K is conditioned** | **opens** a spurious path — this is the trap |

Two rules to memorise: **control forks; never control colliders or mediators.**

In [ ]:
from causal_utils import classify_triple

TRIPLES = [
    ("digital_literacy", "training", "device_age"),
    ("training", "support_calls", "completed"),
    ("training", "completed", "digital_literacy"),
    ("digital_literacy", "completed", "device_age"),
]
RULE = {"collider": "NEVER condition - it OPENS a spurious path",
        "fork": "DO condition - it blocks confounding",
        "chain": "do NOT condition when you want the TOTAL effect",
        "unconnected": "-"}


def show_junctions():
    print(f"{'triple':<52} {'type':<12} rule")
    print("-" * 108)
    for a, b, c in TRIPLES:
        k = classify_triple(G, a, b, c)
        print(f"{a + ' - ' + b + ' - ' + c:<52} {k:<12} {RULE[k]}")
    print("""
Read the four rows carefully:
  row 1  digital_literacy -> training <- device_age   COLLIDER. Two arrows point INTO
         training. `training` is itself a common effect, so conditioning on it would
         link literacy and device age - which is exactly what happens when you analyse
         the enrolled and unenrolled groups separately instead of jointly.
  row 2  training -> support_calls -> completed       CHAIN. One arrow in, one out:
         support_calls is a MEDIATOR carrying part of the effect.
  row 4  digital_literacy -> completed <- device_age  COLLIDER on the OUTCOME. This is
         why analysing only completed sessions is a trap - Step 4 measures the damage.""")


if needs(G):
    show_junctions()

## Step 4 — The collider trap, reproduced *(10 min)*

`collider_sim.parquet` contains two variables generated to be **completely independent**:

- `digital_literacy` — a user trait
- `service_simplicity` — a property of the service they happened to need

and a collider, `completed`, caused by both. Nobody manipulated anything. The only thing that
changes below is **which rows you look at**.

In [ ]:
# TODO: load collider_sim.parquet, then compute
#         corr_uncond = correlation of digital_literacy and service_simplicity, all rows
#         corr_cond   = the same correlation among rows with completed == 1
#       Store them as floats.
coll = None
corr_uncond = None
corr_cond = None

def _c():
    assert corr_uncond is not None and corr_cond is not None, "correlations still None"
    assert abs(corr_uncond) < 0.02, "unconditional correlation should be ~0.00"
    assert corr_cond < -0.10, "conditional correlation should be clearly negative"
check("collider bias reproduced: ~0.00 unconditional -> negative conditional", _c)

if needs(corr_uncond, corr_cond):
    print(f"corr(literacy, simplicity)                 = {corr_uncond:+.4f}")
    print(f"corr(literacy, simplicity | completed = 1) = {corr_cond:+.4f}")

In [ ]:
# TODO: make the before/after scatter. Two panels sharing axes:
#       (left) all 50,000 users with a fitted line; (right) only completed == 1.
#       Sub-sample ~4,000 points per panel for legibility. Annotate both correlations.
print("[ TODO ] collider scatter figure")

## Step 5 — The adjustment-set explorer: more controls is not more rigour *(10 min)*

Now put the theory on the real cohort. The same regression, five different adjustment sets,
five different answers. Only one of them is derived from a graph.

In [ ]:
users = pd.read_csv(DATA / "injaz_users.csv").rename(columns={
    "digital_literacy_score": "digital_literacy",
    "device_age_years": "device_age",
    "enrolled_training": "training",
})
print(f"{len(users):,} users · {users['training'].mean():.1%} enrolled · "
      f"completion {users['completed'].mean():.1%}")
print("\nNote the renames: the DAG's node names now match the dataframe's columns, "
      "\nwhich is exactly the discipline that makes a graph auditable.")

In [ ]:
# TODO: write `effect_under(adjust)` that fits
#         completed ~ training + <adjust...>
#       with HC3 robust SEs and returns a dict with keys
#       {"adjustment_set", "estimate", "ci_low", "ci_high"} (proportions, not pp).
#       Then evaluate it on the five sets in SETS and build `explorer` (a DataFrame).
import statsmodels.formula.api as smf

SETS = {
    "{} - no controls": [],
    "{digital_literacy} only": ["digital_literacy"],
    "{digital_literacy, device_age} - BACKDOOR SET": ["digital_literacy", "device_age"],
    "+ support_calls (MEDIATOR)": ["digital_literacy", "device_age", "support_calls"],
    "kitchen sink (everything available)": ["digital_literacy", "device_age", "support_calls",
                                            "online_filing", "prior_completion_rate",
                                            "age", "prior_sessions"],
}

def effect_under(adjust):
    """TODO: implement - see the docstring requirements above."""
    raise NotImplementedError("effect_under is not implemented yet")


explorer = None      # <-- replace

def _c():
    assert explorer is not None, "explorer is still None"
    bd = explorer.loc[explorer.adjustment_set.str.contains("BACKDOOR"), "estimate"].iat[0]
    none = explorer.loc[explorer.adjustment_set.str.startswith("{} "), "estimate"].iat[0]
    med = explorer.loc[explorer.adjustment_set.str.contains("MEDIATOR"), "estimate"].iat[0]
    assert none > bd, "the unadjusted estimate should be LARGER (confounded upward)"
    assert med < bd, "adding the mediator should SHRINK the estimate"
check("explorer shows confounding upward, mediator downward", _c)

if needs(explorer):
    print(explorer.to_string(index=False))

In [ ]:
# TODO: plot the explorer as a horizontal interval chart, in percentage points,
#       with the planted truth (+6.0 pp) as a reference line and a takeaway annotation.
print("[ TODO ] adjustment-set explorer chart")

## Step 6 — A defended graph beats a tidy one *(8 min)*

You take the minimal DAG to a domain expert at Injaz. She adds three arrows in ninety
seconds:

- *"Older users have lower digital literacy and older devices."* → `age → digital_literacy`,
  `age → device_age`, `age → training`
- *"Heavy prior users hear about the programme through the portal."* → `prior_sessions → training`
- *"Prior completion rate is the best single predictor of completing again."* →
  `prior_sessions → prior_completion_rate → completed`

This is what a DAG is **for**: it moves the causal argument out of the code and onto a
whiteboard where a domain expert can attack it. Re-derive the adjustment set from the new
graph and re-estimate.

In [ ]:
# TODO: build the extended graph `G2` (the 7 original edges plus the 6 new ones listed
#       above), wrap it as `dag2`, derive `minimal_set2`, and re-estimate with
#       effect_under(...). Store the new estimate dict in `eff_defended`.
G2 = None
dag2 = None
minimal_set2 = None
eff_defended = None

def _c():
    assert minimal_set2 is not None, "minimal_set2 is still None"
    assert "support_calls" not in minimal_set2, "the mediator must still be excluded"
    assert eff_defended is not None, "eff_defended is still None"
    assert abs(eff_defended["estimate"] - 0.060) < 0.01, \
        f"expected ~+6.0 pp from the defended graph, got {100*eff_defended['estimate']:+.2f} pp"
check("defended graph recovers the planted +6.0 pp", _c)

### Step 6b — The mediator trap, on the defended graph

Now add `support_calls` to the *correct* adjustment set and watch the effect shrink. This is
the "control that erased the effect" case study, reproduced on your own numbers.

In [ ]:
# TODO: estimate with minimal_set2 PLUS support_calls, store in `eff_mediator`,
#       and print the before/after comparison.
eff_mediator = None

def _c():
    assert eff_mediator is not None, "eff_mediator is still None"
    assert eff_mediator["estimate"] < eff_defended["estimate"], \
        "controlling for the mediator should SHRINK the estimate"
check("mediator trap reproduced: the effect shrinks", _c)

## Step 7 — identify → estimate → refute, end to end *(6 min)*

The DoWhy backbone, executed with `causal_utils`. Three cells, three verbs, one auditable
artefact.

In [ ]:
# ---------------------------------------------------------------- IDENTIFY
if needs(dag2):
    print("=" * 72)
    print("STEP 1 of 3 - IDENTIFY  (read the estimand off the graph)")
    print("=" * 72)
    print(dag2.explain())

In [ ]:
# ---------------------------------------------------------------- ESTIMATE
from causal_utils import match_nearest, ipw_ate

# A SUPERSET of a valid backdoor set is still valid, as long as it adds no
# descendant of the treatment. Check, do not assume.
COV_DF = ["digital_literacy", "device_age", "prior_completion_rate",
          "age", "prior_sessions"]
mres = None
ipw = None


def run_estimates():
    global mres, ipw
    print("=" * 72)
    print("STEP 2 of 3 - ESTIMATE  (adjust for exactly the identified set, no more)")
    print("=" * 72)
    print("  is the estimation covariate set a valid adjustment set? ->",
          is_valid_adjustment_set(G2, "training", "completed", set(COV_DF)))
    print("  (it is a superset of the minimal set and adds no descendant of the treatment)\n")

    mres = match_nearest(users, "training", "completed", COV_DF, caliper=0.02)
    ipw = ipw_ate(users, "training", "completed", COV_DF)

    print(f"  regression on the backdoor set : {100*eff_defended['estimate']:+.2f} pp "
          f"[{100*eff_defended['ci_low']:+.2f}, {100*eff_defended['ci_high']:+.2f}]")
    print(f"  propensity matching (ATT)      : {100*mres.att:+.2f} pp "
          f"[{100*mres.ci_low:+.2f}, {100*mres.ci_high:+.2f}]  "
          f"worst |SMD| after {mres.worst_smd_after:.3f}")
    print(f"  IPW (stabilised)               : {100*ipw['ate']:+.2f} pp "
          f"[{100*ipw['ci_low']:+.2f}, {100*ipw['ci_high']:+.2f}]")
    spread = 100*(max(eff_defended["estimate"], mres.att, ipw["ate"])
                  - min(eff_defended["estimate"], mres.att, ipw["ate"]))
    print(f"\n  Spread across the three: {spread:.2f} pp - well inside every interval.")
    print("  Note what this is and is NOT evidence of. All three rest on the SAME graph,")
    print("  so they share the same blind spot. Agreement here rules out an estimator bug;")
    print("  it says nothing about whether the graph is right. Only Step 8's sensitivity")
    print("  analysis speaks to that.")


if needs(G2, eff_defended):
    run_estimates()

In [ ]:
# ---------------------------------------------------------------- REFUTE
from causal_utils import (placebo_outcome_test, random_common_cause_test,
                          subset_refutation, rosenbaum_bounds)

rb = None
gamma_star = None


def run_refutations():
    global rb, gamma_star
    print("=" * 72)
    print("STEP 3 of 3 - REFUTE  (try to break your own result)")
    print("=" * 72)
    users["distance_km_placebo"] = users["distance_km"]
    ref_placebo = placebo_outcome_test(users, "training", "distance_km_placebo", COV_DF)
    ref_rcc = random_common_cause_test(users, "training", "completed", COV_DF, n_sims=10)
    ref_sub = subset_refutation(users, "training", "completed", COV_DF, frac=0.7, n_sims=10)
    rb = rosenbaum_bounds(mres.matched, "completed",
                          gammas=(1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.5))
    gamma_star = float(rb.loc[rb["still_significant_at_05"], "gamma"].max())

    print(f"  placebo outcome (distance_km) : {ref_placebo['placebo_effect']:+.3f} km, "
          f"p = {ref_placebo['p_value']:.3f}  -> "
          f"{'PASS' if ref_placebo['passes'] else 'FAIL'}")
    print(f"  random common cause           : max shift "
          f"{100*ref_rcc['max_abs_shift']:.4f} pp  -> "
          f"{'PASS' if ref_rcc['passes'] else 'FAIL'}")
    print(f"  70% subsets                   : {100*ref_sub['subset_mean']:+.2f} pp vs "
          f"{100*ref_sub['original']:+.2f} pp  -> "
          f"{'PASS' if ref_sub['passes'] else 'FAIL'}")
    print(f"  Rosenbaum Gamma*              : {gamma_star:.2f}  "
          f"(a hidden confounder stronger than this overturns it)")

    def _c():
        assert ref_placebo["passes"] and ref_rcc["passes"] and ref_sub["passes"], \
            "a refuter failed - investigate before reporting anything"
    check("identify -> estimate -> refute completed", _c)


if needs(mres):
    run_refutations()

### Optional — the same workflow in DoWhy

In [ ]:
# --- OPTIONAL: DoWhy equivalent (safe to run; skips if not installed) -----
GML = """graph [directed 1
  node [id "digital_literacy" label "digital_literacy"]
  node [id "device_age" label "device_age"]
  node [id "prior_completion_rate" label "prior_completion_rate"]
  node [id "training" label "training"]
  node [id "support_calls" label "support_calls"]
  node [id "completed" label "completed"]
  edge [source "digital_literacy" target "training"]
  edge [source "digital_literacy" target "completed"]
  edge [source "device_age" target "training"]
  edge [source "device_age" target "completed"]
  edge [source "prior_completion_rate" target "completed"]
  edge [source "training" target "support_calls"]
  edge [source "support_calls" target "completed"]
  edge [source "training" target "completed"] ]"""

try:
    from dowhy import CausalModel

    model = CausalModel(data=users, treatment="training", outcome="completed", graph=GML)
    estimand = model.identify_effect()
    print(estimand)                       # note: support_calls NOT in the adjustment set
    est = model.estimate_effect(estimand, method_name="backdoor.linear_regression")
    ref = model.refute_estimate(estimand, est, method_name="placebo_treatment_refuter")
    print("Effect:", round(est.value, 4), "| placebo:", ref)

except ImportError:
    print("DoWhy is not installed on this image - expected, and nothing is lost.\n"
          "The GML graph above is the SAME graph you built with networkx; DoWhy would\n"
          "read it and return the same adjustment set that dag2.minimal_adjustment_set()\n"
          "returned. The three verbs were all executed above:\n"
          "  identify -> dag2.explain() / backdoor_adjustment_sets()\n"
          "  estimate -> regression on the identified set, match_nearest, ipw_ate\n"
          "  refute   -> placebo_outcome_test, random_common_cause_test,\n"
          "              subset_refutation, rosenbaum_bounds\n"
          "Keep the GML string: it is a portable, reviewable record of your assumptions.")

## Step 8 — The identification verdict *(6 min)*

For any observational question a professional returns **exactly one of three verdicts**:

1. **Identifiable by backdoor set S** — here is S, here is the estimate, here is the
   sensitivity.
2. **Identifiable by another route** — frontdoor, or an instrument (Lab 5's IV).
3. **Not identifiable with current data** — and here is precisely what is missing.

Anything else — running a regression and hoping — is not a verdict.

In [ ]:
def render_verdict():
    return f"""
IDENTIFICATION VERDICT - effect of the digital-literacy training programme on completion
-----------------------------------------------------------------------------------------
VERDICT      (1) IDENTIFIABLE by the backdoor criterion.

ADJUSTMENT   {sorted(minimal_set2)}
             derived from the defended DAG, not chosen by hand.

EXCLUDED     support_calls  - descendant of the treatment (mediator, and in this
                              cohort also a collider). Adding it costs
                              {100*(eff_defended['estimate'] - eff_mediator['estimate']):.2f} pp
                              of the effect.
             online_filing  - a separate treatment with its own confounding; not
                              on any backdoor path for THIS question.

ESTIMATE     {100*eff_defended['estimate']:+.2f} pp
             [{100*eff_defended['ci_low']:+.2f}, {100*eff_defended['ci_high']:+.2f}]
             (matching {100*mres.att:+.2f} pp, IPW {100*ipw['ate']:+.2f} pp - all on the
             same identification argument)

WHAT WOULD   An unmeasured common cause of enrolling AND completing. Personal
BREAK IT     MOTIVATION is the obvious candidate: motivated users enrol and
             motivated users complete, and Injaz measures neither. Rosenbaum
             bounds put the tolerance at Gamma* = {gamma_star:.2f} - fairly weak.
             If motivation moves the odds of enrolling by more than {gamma_star:.2f}x,
             this verdict flips to (3) NOT IDENTIFIABLE.

WHAT WOULD   A randomised invitation to the programme. Six weeks, no extra
SETTLE IT    programme cost, and Gamma becomes 1 by construction.
"""


if needs(minimal_set2, eff_defended, eff_mediator, mres, ipw, gamma_star):
    VERDICT = render_verdict()
    print(VERDICT)

## Step 9 — Compare against the planted ground truth

In [ ]:
import json
GT = json.loads((DATA / "GROUND_TRUTH.json").read_text())


def ground_truth_comparison():
    g5, g6 = GT["module5_user_cohort"], GT["module6_collider"]
    rows = [
        {"quantity": "collider corr, unconditional", "recovered": corr_uncond,
         "planted": g6["corr_unconditional"]},
        {"quantity": "collider corr | completed == 1", "recovered": corr_cond,
         "planted": g6["corr_conditional_on_collider"]},
        {"quantity": "training effect, no controls (pp)",
         "recovered": 100*explorer["estimate"].iat[0],
         "planted": 100*g5["naive_diff_training"]},
        {"quantity": "training effect, defended backdoor set (pp)",
         "recovered": 100*eff_defended["estimate"], "planted": 100*g5["true_ATT_training"]},
        {"quantity": "training effect, mediator controlled (pp)",
         "recovered": 100*eff_mediator["estimate"], "planted": float("nan")},
    ]
    out = pd.DataFrame(rows)
    out["gap"] = out["recovered"] - out["planted"]
    print(out.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
    print("\n(the mediator row has no planted counterpart: it estimates a DIRECT effect,")
    print(" which is a different quantity from the +6.0 pp total effect that was planted)")

    def _c():
        assert abs(corr_uncond - g6["corr_unconditional"]) < 0.01, "collider baseline mismatch"
        assert abs(corr_cond - g6["corr_conditional_on_collider"]) < 0.01, \
            "conditional collider correlation mismatch"
        assert abs(eff_defended["estimate"] - g5["true_ATT_training"]) < 0.005, \
            "the defended-graph estimate should land within 0.5 pp of the planted +6.0 pp"
    check("collider and backdoor results match the answer key", _c)
    return out


if needs(corr_uncond, corr_cond, explorer, eff_defended, eff_mediator):
    comparison = ground_truth_comparison()

## What you now own

In [ ]:
def build_artefact():
    return {
        "notebook": "lab6 - DAGs, colliders and defensible identification",
        "reusable_pipeline": [
            "1. draw the DAG BEFORE the model; one-line justification per arrow",
            "2. dag.explain() / backdoor_adjustment_sets() -> derive, do not choose",
            "3. classify every candidate control as fork / chain / collider",
            "4. estimate on the identified set only",
            "5. show what the wrong sets do (explorer table + chart)",
            "6. take the graph to a domain expert; re-derive; re-estimate",
            "7. refute + sensitivity",
            "8. issue ONE of the three identification verdicts, with the breaking variable",
        ],
        "portable_artefacts": ["the DAG figure", "the GML string (reviewable assumptions)",
                               "the adjustment-set explorer table", "the collider scatter",
                               "the identification verdict"],
        "numbers_recovered": {
            "collider_corr_unconditional": round(corr_uncond, 4),
            "collider_corr_conditional": round(corr_cond, 4),
            "minimal_graph_backdoor_set": sorted(minimal_set),
            "defended_graph_backdoor_set": sorted(minimal_set2),
            "effect_no_controls_pp": round(100*explorer["estimate"].iat[0], 2),
            "effect_defended_set_pp": round(100*eff_defended["estimate"], 2),
            "effect_with_mediator_pp": round(100*eff_mediator["estimate"], 2),
            "gamma_star": gamma_star,
        },
        "rules_you_can_now_state": [
            "Control forks. Never control colliders or mediators.",
            "The adjustment set is DERIVED from a graph, never chosen by instinct.",
            "Filtering rows is conditioning: sample selection is collider conditioning.",
            "More controls is not more rigour - it is often more bias.",
            "If no measured set closes the backdoors, say 'not identifiable' and name "
            "what is missing.",
        ],
    }


if needs(corr_uncond, minimal_set, minimal_set2, eff_defended, eff_mediator):
    ARTEFACT = build_artefact()
    for k, v in ARTEFACT.items():
        print(f"\n{k}:")
        print("  " + (("\n  ".join(str(x) for x in v)) if isinstance(v, list) else str(v)))

## Mini exercises

### 1. Debugging — the kitchen-sink regression *(6 min)*

A colleague's model controls for `support_calls` (a mediator) **and** restricts the sample to
`completed == 1` users when computing a covariate correlation (a collider condition). Use the
DAG to remove both errors and recover the correct effect.

In [ ]:
# TODO: (a) name each of the two errors using the DAG vocabulary,
#       (b) produce the corrected estimate, and
#       (c) write DIAGNOSIS explaining what each error does to the answer.
corrected = None
DIAGNOSIS = ""

if needs(corrected):
    print(f"corrected estimate: {100*corrected['estimate']:+.2f} pp")
    print(DIAGNOSIS)

### 2. DAG-drawing exercise *(8 min, in pairs)*

Draw the DAG for the **IV analysis from Lab 5** — the effect of `online_filing` on
`completed`, instrumented by `log_distance_km` — and show *graphically* why distance
qualifies as an instrument.

An instrument is a node with (a) an arrow into *T*, (b) **no** arrow into *Y* except through
*T*, and (c) no shared confounder with *Y*.

In [ ]:
# TODO: build `G_iv` with nodes: log_distance, online_filing, completed,
#       motivation (UNMEASURED confounder of filing and completion), digital_literacy.
#       Then verify with the graph that:
#         - there is NO directed path log_distance -> ... -> completed except via online_filing
#         - motivation is a common cause of online_filing and completed
G_iv = None

def _c():
    assert G_iv is not None, "G_iv is still None"
    paths = list(nx.all_simple_paths(G_iv, "log_distance", "completed"))
    assert all("online_filing" in p for p in paths), \
        "exclusion restriction violated: distance reaches the outcome without going through filing"
check("distance satisfies the exclusion restriction in your graph", _c)

### 3. Discussion *(5 min)*

**(a)** A colleague says *"more controls always reduce bias."* Refute it with a collider
example in two sentences.

**(b)** When is deliberately controlling for a mediator the *right* thing to do?

In [ ]:
DISCUSSION = {
    "a_more_controls_always_better": "TODO",
    "b_when_is_a_mediator_control_right": "TODO",
}
for k, v in DISCUSSION.items():
    print(f"\n{k}:\n  {v}")

---

### Quiz recall (Module 6)

1. **What does an *absent* arrow assert?** → that there is no direct causal effect between
   those variables.
2. **What happens when you condition on a collider?** → a spurious association opens between
   its causes.
3. **State the backdoor criterion.** → adjust for a set that blocks all backdoor paths and
   contains no descendant of the treatment.
4. **Why not control for a mediator when estimating a total effect?** → it blocks part of the
   causal path, shrinking the effect.
5. **What are the three identification verdicts?** → identifiable by backdoor set *S*;
   identifiable by another route (frontdoor / IV); not identifiable with current data.

### Next

**Lab 7** — everything at once. One real decision, method selection, a frozen plan, an A/B
test, a reconciliation with the observational evidence, and a memo a director can act on.